In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# load cereals csv
cereals = pd.read_csv(
    "../data/raw/Cereals_Farmgate_Prices.csv",
    skiprows=1,
    na_values = [".."]
)

print(f"{cereals.shape}\n")
print(f"{cereals.info()}\n")
print(f"{cereals.columns}\n")

# Count missing values
# missing_values = cereals.iloc[:, 2:].isna().sum().sum()

# Print missing months with missing values
# missing_per_month = cereals.iloc[:, 2:].isna().sum()
# print(missing_per_month[missing_per_month > 0])


#Missing prices per commodity
print("\n--Missing prices per commodity")
missing_by_commodity = cereals.iloc[:, 2:].isna().sum(axis=1)
print(f"{
        pd.DataFrame({
            "Commodity": cereals["Commodity"],
            "Missing Months": missing_by_commodity
        })
    }\n"
)


# identify missing months for each commodity
print("\n--Missing months for each commodity")
for index, row in cereals.iterrows():
    monthly_prices = row.iloc[2:]
    missing_prices = monthly_prices.isna()
    
    # Get the month names with missing prices
    missing_months = monthly_prices[missing_prices].index.tolist()

    # Display the result
    print(f"{row['Commodity']}: {missing_months}")

In [ ]:
#4. DATA TRANSFORMATION

monthly_columns = cereals.columns[2:]

# Convert monthly columns into rows
cereals_long = cereals.melt(
    id_vars = ['Geolocation', 'Commodity'],
    value_vars = monthly_columns,
    var_name = 'Date',
    value_name = 'Price'
)

# Assign commodity category
cereals_long["Category"] = "Cereals"

# Convert Date to proper format
cereals_long['Date'] = pd.to_datetime(
    cereals_long['Date'],
    format = "%Y %B"
)

# Sort records by commodity and date
cereals_long = cereals_long.sort_values(
    ['Commodity', 'Date']
).reset_index(drop=True)

# Arrange columns for preferred orderrr
cereals_long = cereals_long[
    ['Geolocation', 'Category', 'Commodity', 'Date', 'Price']
]

cereals_long


In [ ]:
# Transformation of 7 csv to 1 dataframe

from pathlib import Path
transformed_datasets = [] # list of transfromed csv

# Find csv path
csv_files = sorted(Path('../data/raw').glob("*.csv")) 

# Process for transfromting CSV
for file in csv_files:
    # Get category name from filename
    category = file.stem.replace('_Farmgate_Prices', "").replace("_", " ")
    
    # read csv file
    data = pd.read_csv(file, skiprows=1, na_values=['..'])
    
    # get only month columns
    monthly_columns = data.columns[2:] 
    
    # convert wide to long format
    data_long = data.melt(
        id_vars=['Geolocation', 'Commodity'],
        value_vars = monthly_columns,
        var_name = 'Date',
        value_name = 'Price'
    )
    data_long['Category'] = category #added commodity category column
    
    
    # Convert Date values into propoer date format
    data_long['Date'] = pd.to_datetime(
        data_long['Date'],
        format = "%Y %B"
    )
    
    # sort records by commodity and datee
    data_long = data_long.sort_values(
        ['Category', 'Commodity', 'Date']
    ).reset_index(drop=True)
    
    
    # arrange columns to preferred format
    data_long = data_long[
        ['Geolocation', 'Category', 'Commodity', 'Date', 'Price']
    ]
    
    # store transformed dataset in a list
    transformed_datasets.append(data_long)

    print(f"Wide Format: {category}, {data.shape}")
    print(f"Long Format: {category}, {data_long.shape}\n")

In [ ]:
#combined all seven transformed dataframes
combined_data = pd.concat(
    transformed_datasets,
    ignore_index = True
)

# Shapes and number of commodity per category
print(combined_data.shape)
print("Total Categories:", combined_data["Category"].nunique(), "\n")
print(combined_data.groupby("Category")["Commodity"].nunique())

In [ ]:
#5. Data Quality Assessment
missing_values = combined_data.isna().sum()
print(f"{missing_values}\n")

# count missing prices each category
missing_by_category = (
    combined_data["Price"].isna()
    .groupby(combined_data["Category"])
    .sum()
)
print(f"{missing_by_category}\n")

# count missing prices per commodity
missing_by_commodity = (
    combined_data['Price'].isna()
    .groupby([combined_data['Category'], combined_data['Commodity']])
    .sum()
)
print(f"{missing_by_commodity}")

# Find the latest available price date per commodity
latest_dates = (
    combined_data.dropna(subset=["Price"])
    .groupby(["Category", "Commodity"])["Date"]
    .max()
)
print(latest_dates)

In [ ]:
# Convert latest dates into a table
latest_dates_table = latest_dates.reset_index(name="Latest_Date")

# Match each record with its commodity's latest date
missing_check = combined_data.merge(
    latest_dates_table,
    on=["Category", "Commodity"]
)

# Find missing prices within recorded history
historical_gaps = missing_check[
    missing_check["Price"].isna() &
    (missing_check["Date"] <= missing_check["Latest_Date"])
]

historical_gaps[["Category", "Commodity", "Date", "Price"]]

In [ ]:
# Find missing prices after the latest recorded date
trailing_gaps = missing_check[
    missing_check["Price"].isna() &
    (missing_check["Date"] > missing_check["Latest_Date"])
]

trailing_gaps[["Category", "Commodity", "Date", "Price"]]

In [ ]:
# Identify prices equal to zero
zero_prices = combined_data[combined_data["Price"] == 0]

print("Total zero prices:", len(zero_prices))
zero_prices

In [ ]:
# Check duplicate records for the same commodity and month
duplicates = combined_data.duplicated(
    subset=["Geolocation", "Category", "Commodity", "Date"]
).sum()

print("Total duplicate records:", duplicates)

In [171]:
# Count prices below zero
negative_prices = (combined_data["Price"] < 0).sum()

print("Total negative prices:", negative_prices)

Total negative prices: 0


In [ ]:
# DESCRIPTIVE STATISTICS (EDA)

# Descriptive statistics for each commodity
commodity_stats = combined_data.groupby(
    ["Category", "Commodity"]
)["Price"].describe()

commodity_stats

In [ ]:
# Calculate the IQR for each commodity
commodity_stats["IQR"] = commodity_stats["75%"] - commodity_stats["25%"]

commodity_stats[["25%", "75%", "IQR"]]

# Calculate lower and upper outlier boundaries
commodity_stats["Lower_Bound"] = commodity_stats["25%"] - 1.5 * commodity_stats["IQR"]
commodity_stats["Upper_Bound"] = commodity_stats["75%"] + 1.5 * commodity_stats["IQR"]

# Display the calculated boundaries
commodity_stats[["Lower_Bound", "Upper_Bound"]]

# Match each commodity's outlier boundaries to its monthly prices
outlier_check = combined_data.merge(
    commodity_stats[["Lower_Bound", "Upper_Bound"]].reset_index(),
    on=["Category", "Commodity"],
    how="left"
)

# Identify prices outside their commodity's IQR boundaries
potential_outliers = outlier_check[
    (outlier_check["Price"] < outlier_check["Lower_Bound"]) |
    (outlier_check["Price"] > outlier_check["Upper_Bound"])
]

print("Total potential outliers:", len(potential_outliers))

potential_outliers.head(10)

In [ ]:
# Count potential outliers for each commodity
outliers_by_commodity = (
    potential_outliers.groupby(["Category", "Commodity"])
    .size()
    .sort_values(ascending=False)
)

print(outliers_by_commodity)

In [170]:
# Save the combined dataset for data cleaning
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

combined_data.to_csv(
    "../data/processed/combined_farmgate_prices.csv",
    index=False
)